# Modelling: MLP

This notebook reproduces the exact same clean + split from `04_modelling_classical.ipynb` (same `dataset_malwares.csv`, same `CORE_TRAITS` columns, same `RANDOM_STATE`), then trains a small Multi-Layer Perceptron as a deep-learning comparison point against Random Forest and XGBoost. This model is trained and evaluated for comparison only, `06_evaluation.ipynb` decides which of the three candidates is actually deployed.

The test set is not touched here, only validation, matching `04_modelling_classical.ipynb`.

**Note:** this notebook trains on `dataset_malwares.csv` directly, the original, unaugmented dataset, matching `04_modelling_classical.ipynb`'s data source (see that notebook's intro for why real-file augmentation was removed from this project). Cleaning and model-building are done step by step below rather than through helper functions, so each step's effect is visible on its own. Every cell from "Load, Clean, and Reproduce the Split" onward needs a fresh **Restart & Run All** before its output can be trusted.

In [1]:
# Standard library
import sys

# Third-party
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Model, layers

pd.set_option("display.max_columns", None)
sys.path.append("../src")

from constants import CORE_TRAITS, ID_COLUMNS, LABEL_COLUMN, RANDOM_STATE

# Seed TensorFlow's own randomness (weight initialization, shuffling), separate
# from RANDOM_STATE, which only seeds scikit-learn/XGBoost, so MLP training is
# reproducible too, not just the classical models.
tf.random.set_seed(RANDOM_STATE)

## Load, Clean, and Reproduce the Split

The same load, feature selection, missing-value check, and duplicate check from `04_modelling_classical.ipynb`, repeated here step by step against the same `dataset_malwares.csv` with the same `CORE_TRAITS` columns, then the same two `train_test_split` calls. Since the same fixed `RANDOM_STATE` is used on the same input file, this reproduces the exact same train/validation/test rows `04_modelling_classical.ipynb` used, without saving split files to disk.

In [2]:
df = pd.read_csv("../data/dataset_malwares.csv")
X = df[CORE_TRAITS].copy()
y = df[LABEL_COLUMN].copy()

# Drop rows with missing values
mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]

# Drop duplicate feature rows
dup_mask = ~X.duplicated()
X, y = X[dup_mask], y[dup_mask]

print(f"{len(X)} rows after cleaning.")

13517 rows after cleaning.


In [3]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE)
print(f"Train: {len(y_train)}, Validation: {len(y_val)}, Test: {len(y_test)}")

Train: 9461, Validation: 2028, Test: 2028


## Scale the Features

The scaler is fit once on `X_train` only, the same leakage-safe rule `04_modelling_classical.ipynb`'s pipelines enforce automatically; here it has to be done by hand since a neural network is not wrapped in a scikit-learn `Pipeline`.

In [4]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)

## Build the MLP

Input -> 128 neurons -> 64 neurons -> 1 output (probability of malicious), built directly rather than inside a function since it's only constructed once. `Dense` = fully-connected layer. `relu` lets it learn non-linear patterns. `Dropout` randomly disables 30% of neurons during training to stop it memorising the data (overfitting). `sigmoid` squashes the final output into a 0-1 probability.

In [5]:
inputs = layers.Input(shape=(X_train_scaled.shape[1],))
x = layers.Dense(128, activation="relu")(inputs)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 10)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │           1,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 9,729 (38.00 KB)

 Trainable params: 9,729 (38.00 KB)

 Non-trainable params: 0 (0.00 B)

## Handle Class Imbalance

Computed from the actual training counts, not assumed: `CORE_TRAITS` cleaning found more malicious rows than benign, the minority class (benign) is upweighted so the network cannot reach a good loss just by favouring malicious.

In [6]:
class_counts = y_train.value_counts()
majority_label = class_counts.idxmax()
minority_label = class_counts.idxmin()
class_weight = {
    int(majority_label): 1.0,
    int(minority_label): float(class_counts[majority_label] / class_counts[minority_label]),
}
print("Using class_weight:", class_weight)

Using class_weight: {1: 1.0, 0: 1.796630209872894}


## Train the Model

`EarlyStopping` stops training once validation performance stalls for 3 epochs, and rolls back to the best weights seen rather than the last epoch's.

In [7]:
early_stop = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train.values,
    validation_data=(X_val_scaled, y_val.values),
    epochs=30, batch_size=64,
    callbacks=[early_stop], class_weight=class_weight, verbose=2,
)

Epoch 1/30
148/148 - 3s - 24ms/step - accuracy: 0.7095 - loss: 0.7304 - val_accuracy: 0.7613 - val_loss: 0.4863
Epoch 2/30
148/148 - 0s - 2ms/step - accuracy: 0.7534 - loss: 0.6332 - val_accuracy: 0.7929 - val_loss: 0.4404
Epoch 3/30
148/148 - 0s - 2ms/step - accuracy: 0.7705 - loss: 0.5858 - val_accuracy: 0.8111 - val_loss: 0.4120
Epoch 4/30
148/148 - 0s - 2ms/step - accuracy: 0.7808 - loss: 0.5721 - val_accuracy: 0.8190 - val_loss: 0.3966
Epoch 5/30
148/148 - 0s - 2ms/step - accuracy: 0.7908 - loss: 0.5541 - val_accuracy: 0.8220 - val_loss: 0.3909
Epoch 6/30
148/148 - 1s - 5ms/step - accuracy: 0.7984 - loss: 0.5410 - val_accuracy: 0.8393 - val_loss: 0.3719
Epoch 7/30
148/148 - 1s - 4ms/step - accuracy: 0.8081 - loss: 0.5328 - val_accuracy: 0.8412 - val_loss: 0.3656
Epoch 8/30
148/148 - 1s - 4ms/step - accuracy: 0.8123 - loss: 0.5184 - val_accuracy: 0.8397 - val_loss: 0.3625
Epoch 9/30
148/148 - 0s - 2ms/step - accuracy: 0.8214 - loss: 0.5182 - val_accuracy: 0.8462 - val_loss: 0.3550


## Evaluate on the Validation Set

>= 0.5 turns the raw probability into a final verdict, the same threshold convention the classical models use.

In [8]:
val_preds = (model.predict(X_val_scaled) >= 0.5).astype(int).ravel()
print(classification_report(y_val, val_preds, target_names=["benign", "malicious"]))

64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
              precision    recall  f1-score   support

      benign       0.79      0.91      0.85       725
   malicious       0.95      0.87      0.91      1303

    accuracy                           0.88      2028
   macro avg       0.87      0.89      0.88      2028
weighted avg       0.89      0.88      0.89      2028



## Save the MLP Candidate

Saved in Keras's own native format (not joblib), the same way `train_mlp.py` always saved it, so `06_evaluation.ipynb` can reload it without needing a custom wrapper class (see the earlier discussion on why `LstmPipeline`-style wrappers are only needed when joblib has to serialize a non-scikit-learn model).

In [9]:
model.save("../models/mlp_comparison.keras")
print("Saved to ../models/mlp_comparison.keras")

Saved to ../models/mlp_comparison.keras


## Summary

- Reproduces `04_modelling_classical.ipynb`'s exact split on `dataset_malwares.csv` (same `RANDOM_STATE`, same `CORE_TRAITS` columns), the cleaning steps repeated inline: 13,517 rows after cleaning, confirmed via the real, executed output above, matching `04_modelling_classical.ipynb`'s own cleaning result exactly.
- Added `tf.random.set_seed(RANDOM_STATE)`, since TensorFlow has its own separate randomness (weight initialization) that `RANDOM_STATE` alone does not reach, unlike scikit-learn/XGBoost where it's an explicit parameter.
- Builds a small MLP (128 -> 64 -> 1, dropout 0.3, 9,729 parameters) directly, no helper function, architecture confirmed via the real `model.summary()` output above (unaffected by the data-source change, since `CORE_TRAITS` is always 10 input features either way).
- Trains with class weighting to offset the training set's malicious majority, using weights computed fresh from the actual split, not hardcoded.
- **TODO before submission:** this notebook needs a fresh Restart & Run All (the data source is now `dataset_malwares.csv`, not the removed `dataset_augmented.csv`), then replace this bullet with the actual validation accuracy/macro-F1 numbers and compare them against `04_modelling_classical.ipynb`'s Random Forest and XGBoost results on the same data.
- Saves the trained model to `models/mlp_comparison.keras` using Keras's native format, for `06_evaluation.ipynb` to reload alongside the two classical candidates.